In [1]:
from IPython.display import Image

> 计算单元；

- cuda cores
    - for GPGPU（General purpose GPU）
        - 实际上是 GPU 内部最微小的算术逻辑单元（ALU），它负责执行最基础的单点浮点运算。Nvidia 将成百上千个这样的微小 ALU 打包成一个“流多处理器”（SM - Streaming Multiprocessor）。以 RTX 5090 为例，其内部实际上是 170 个 SM。(21760 / 170 = 128)
    - 一个 CUDA Core 擅长执行基础的浮点（FP32/FP64）或整数数学运算。(scalar 级别的）
        - `c[i] = a[i] * b[i] + d[i]`
    - 一个典型的 CUDA Core 通常包含用于浮点运算（FP32）或整数运算（INT32）的 **FMA**（Fused Multiply-Add）单元（内含 ALU）。
        - 它的特点是“一维”的：在一个时钟周期内，一个 CUDA Core 只能为一个线程处理一个基础的数据操作（例如执行 $A \times B + C$）。它负责处理图形渲染以及深度学习中的非线性操作（如 ReLU 激活、Layer Normalization、向量点积等）。
- tensor cores
    - https://www.youtube.com/watch?v=Cak8ASX7NOk
    - https://www.youtube.com/watch?v=fqQyopIWN-8
    - Tensor Core 是自 Volta 架构起引入的专用硬件单元。如果说 CUDA Core 执行的是一维的标量计算，Tensor Core 则是实现了“降维打击”的二维矩阵计算。
        - 在当前的大型语言模型（LLM）训练和推理中，绝大部分的算力（TFLOPS）皆由 Tensor Core 提供。
    - 矩阵乘加（Matrix Multiply-Accumulate, MMA）：形如 $D = A \times B + C$
        - 大矩阵乘法 C = A×B + C 是由无数个 tile 的 **MMA** 累起来拼出来的。
        - 它可以在单个时钟周期内，直接吞下一个 $4 \times 4$ 的数据块，瞬间完成 64 次乘法和 64 次加法。
            - $D_{1,1} = (A_{1,1} \times B_{1,1}) + (A_{1,2} \times B_{2,1}) + (A_{1,3} \times B_{3,1}) + (A_{1,4} \times B_{4,1}) + C_{1,1}$

In [2]:
Image(url='./imgs/gpu-sm-cores.png', width=500)

- SM 管理和调度，CUDA Core / Tensor Core 真正干活。
    - SM：是一个较大的执行/调度单元
    - CUDA Core：是 SM 里的普通数值计算执行单元
    - Tensor Core：是 SM 里的矩阵计算专用单元
    - ALU：是一个更泛化的术语，意思是“算术逻辑单元”